## Считаем метрики по оптимизированному роботу

In [77]:
import pandas as pd
import numpy as np
import random
import os
import requests

# ⏳ прогресс-бары
from tqdm import tqdm

# загрузка моделей и функци для предобработки данных
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report,roc_auc_score

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.width', 0)
pd.set_option('display.max_colwidth', None)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [78]:
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

In [79]:
data_download_income_new_nan_classified = pd.read_csv('data_download_new_nan_classified.csv')
data_download_income_new =  pd.read_csv('../data_download_new.csv')

In [80]:
data_download_income_new_nan_classified.id.count()

24888

In [81]:
data_download_income_new.id.count()

24888

In [82]:
data_download_income_new.id.equals(data_download_income_new_nan_classified.id)

True

In [83]:
data_download_income_new_nan_classified['predicted_article'].isna().sum()

0

In [84]:
y_true = data_download_income_new['articles__name']
y_pred = data_download_income_new_nan_classified['predicted_article']

In [85]:
# считаем метрики 

accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred, average='weighted')
recall = recall_score(y_true, y_pred, average='macro')
f1 = f1_score(y_true, y_pred, average='weighted')

print("Accuracy:", accuracy)
print("Precision (weighted):", precision)
print("Recall (macro):", recall)
print("F1-score (weighted):", f1)

print("\nОтчет по классам:")
print(classification_report(y_true, y_pred))

/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Accuracy: 0.8369093539054966
Precision (weighted): 0.8903613811851631
Recall (macro): 0.5897718656947234
F1-score (weighted): 0.8347083862904646

Отчет по классам:


/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


                                                                       precision    recall  f1-score   support

                                                                 !ФОТ       1.00      1.00      1.00         7
                                                  "НЕСКУЧАЮЩИЕ РУЧКИ"       1.00      1.00      1.00         1
                                                % на остаток по счету       1.00      1.00      1.00        42
                                        % на остаток по счету\депозит       1.00      1.00      1.00         1
                                                        % по депозиту       1.00      1.00      1.00         6
                                                        % с депозитов       1.00      1.00      1.00         1
                                                       02.1. Юр. лица       1.00      1.00      1.00         2
                                        06.1. Заработная плата (штат)       0.86      1.00      0.92         6


/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Recall and F-score are ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/envs/datasphere_cli_env/lib/python3.10/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, l

In [86]:
# считаем метрики

# 1) выравниваем truth/pred по id
df = (
    data_download_income_new[
        ['id', 'articles__name']
    ].merge(
        data_download_income_new_nan_classified[
            ['id', 'predicted_article', 'decision_stage', 'suggestion_type']
        ],
        on='id',
        how='inner'
    )
)
print("before:", len(df))
df = df.drop_duplicates()
print("after:", len(df))


# 2) без доп. чистки: предполагаем, что пропуски уже удалены заранее
coverage = (df['predicted_article'] != '').mean()

# если хотите считать метрики только там, где есть предсказание
eval_df = df[df['predicted_article'] != ''].copy()

y_true = eval_df['articles__name']
y_pred = eval_df['predicted_article']

print(f"Rows total: {len(df)}")
print(f"Rows with prediction: {len(eval_df)}")
print(f"Coverage: {coverage:.4f}")

print(f"Accuracy: {accuracy_score(y_true, y_pred):.4f}")
print(f"F1 macro: {f1_score(y_true, y_pred, average='macro', zero_division=0):.4f}")
print(f"F1 weighted: {f1_score(y_true, y_pred, average='weighted', zero_division=0):.4f}")

# 3) компактная сводка ошибок (вместо полного classification_report)
# Ошибки
err = eval_df[eval_df['articles__name'] != eval_df['predicted_article']].copy()

# 1) Топ-20 пар по ОБЩЕМУ числу ошибок (все стадии вместе)
top_pairs = (
    err.groupby(['articles__name', 'predicted_article'], as_index=False)
    .size()
    .rename(columns={'size': 'total_errors'})
    .sort_values('total_errors', ascending=False)
    .head(20)
)

# 2) Разбивка этих же top-пар по стадиям
stage_split = (
    err.groupby(['articles__name', 'predicted_article', 'decision_stage'], as_index=False)
    .size()
    .rename(columns={'size': 'stage_errors'})
)

# 3) Склеиваем и делаем pivot (колонки = стадии)
pivot_stage = (
    top_pairs.merge(
        stage_split,
        on=['articles__name', 'predicted_article'],
        how='left'
    )
    .pivot_table(
        index=['articles__name', 'predicted_article', 'total_errors'],
        columns='decision_stage',
        values='stage_errors',
        aggfunc='sum',
        fill_value=0
    )
    .reset_index()
    .sort_values('total_errors', ascending=False)
)

print("\nTop-20 ошибок (сумма по всем стадиям) + разбивка по стадиям:")
print(pivot_stage.to_string(index=False))



# 4) метрики по decision_stage
def stage_metrics(g: pd.DataFrame) -> pd.Series:
    return pd.Series({
        'rows': len(g),
        'accuracy': accuracy_score(g['articles__name'], g['predicted_article']),
        'f1_macro': f1_score(g['articles__name'], g['predicted_article'], average='macro', zero_division=0),
        'f1_weighted': f1_score(g['articles__name'], g['predicted_article'], average='weighted', zero_division=0),
    })

by_stage = (
    eval_df.groupby('decision_stage', dropna=False)
    .apply(stage_metrics)
    .reset_index()
    .sort_values('rows', ascending=False)
)

# Справочник стадий
stage_desc = {
    "stage1_own_rules_own_priority": "Этап 1: сработали свои ключевые слова (финальный приоритет)",
    "stage1_own_history_own_priority": "Этап 1: сработало точное/близкое совпадение по своей истории (purpose_norm)",
    "stage1_foreign_rules": "Этап 1: найдено совпадение по чужим ключевикам (обычно как кандидат, не финал)",
    "stage2_history_own_priority": "Этап 2: векторный поиск по нормализованным платежам всей базы, победил свой результат",
    "stage2_keywords_own_priority": "Этап 2: векторный поиск по ключевикам всей базы, победил свой результат",
    "stage3_foreign_to_own_mapping": "Этап 3: чужая категория сопоставлена с ближайшей своей категорией",
    "stage3_majority_vote": "Этап 3: fallback-голосование по кандидатам (обычно new_article_candidate)",
    "no_candidates": "Кандидаты не найдены",
    "stage3_failed": "Финальный выбор не выполнен",
}

# Полный список стадий, которые хотим видеть в отчете всегда
all_stages = [
    "stage1_own_rules_own_priority",
    "stage1_own_history_own_priority",
    "stage2_history_own_priority",
    "stage2_keywords_own_priority",
    "stage3_foreign_to_own_mapping",
    "stage3_majority_vote",
    "no_candidates",
    "stage3_failed",
]

# by_stage у вас уже посчитан выше (groupby + метрики)
by_stage_full = (
    by_stage.set_index("decision_stage")
    .reindex(all_stages)
    .reset_index()
)

for c in ["rows", "accuracy", "f1_macro", "f1_weighted"]:
    by_stage_full[c] = by_stage_full[c].fillna(0)

by_stage_full["comment"] = by_stage_full["decision_stage"].map(stage_desc).fillna("Прочая/нестандартная стадия")

print("\nMetrics by decision_stage (с пояснениями):")
print(by_stage_full.to_string(index=False))


# 5) (опционально) метрики по suggestion_type
by_type = (
    eval_df.groupby('suggestion_type', dropna=False)
    .apply(stage_metrics)
    .reset_index()
    .sort_values('rows', ascending=False)
)

print("\nMetrics by suggestion_type:")
print(by_type.to_string(index=False))


before: 25002
after: 24837
Rows total: 24837
Rows with prediction: 24837
Coverage: 1.0000
Accuracy: 0.8368
F1 macro: 0.5737
F1 weighted: 0.8346

Top-20 ошибок (сумма по всем стадиям) + разбивка по стадиям:
                             articles__name                           predicted_article  total_errors  stage1_own_history_own_priority  stage1_own_rules_own_priority  stage2_history_own_priority  stage3_majority_vote
            Пожертвование на расчетный счет                                    Миксплат          1011                                0                              0                            0                  1011
                                        ФОТ                            Страховые взносы           348                                2                            338                            8                     0
                                   ФОТ - зп                        ФОТ НДФЛ (разобрать)           274                                0         

In [61]:
dup1 = data_download_income_new['id'].value_counts()
dup1 = dup1[dup1 > 1]

dup2 = data_download_income_new_nan_classified['id'].value_counts()
dup2 = dup2[dup2 > 1]

print("Duplicates in first df:", len(dup1))
print("Duplicates in second df:", len(dup2))

Duplicates in first df: 41
Duplicates in second df: 41


In [65]:
display(data_download_income_new.id.nunique())

data_download_income_new.id.duplicated().sum()

20130

45

In [63]:
data_download_income_new_nan_classified.id.nunique()

20130